In [2]:
import pandas as pd
df = pd.read_json("../data/financebench_merged.jsonl", lines=True)
df.shape
df.columns
df.head()

,financebench_id,company,doc_name,question_type,question_reasoning,domain_question_num,question,answer,justification,dataset_subset_label,evidence,gics_sector,doc_type,doc_period,doc_link
0,financebench_id_03029,3M,3M_2018_10K,metrics-generated,Information extraction,None,What is the FY2018 capital expenditure amount ...,$1577.00,The metric capital expenditures was directly e...,OPEN_SOURCE,[{'evidence_text': 'Table of Contents 3M Comp...,Industrials,10k,2018,https://investors.3m.com/financials/sec-filing...
1,financebench_id_04672,3M,3M_2018_10K,metrics-generated,Information extraction,None,Assume that you are a public equities analyst....,$8.70,"The metric ppne, net was directly extracted fr...",OPEN_SOURCE,[{'evidence_text': 'Table of Contents 3M Comp...,Industrials,10k,2018,https://investors.3m.com/financials/sec-filing...
2,financebench_id_00499,3M,3M_2022_10K,domain-relevant,Logical reasoning (based on numerical reasoning),dg06,Is 3M a capital-intensive business based on FY...,"No, the company is managing its CAPEX and Fixe...",CAPEX/Revenue\nFixed Assets/Total Assets\nROA=...,OPEN_SOURCE,[{'evidence_text': '3M Company and Subsidiarie...,Industrials,10k,2022,https://investors.3m.com/financials/sec-filing...
3,financebench_id_01226,3M,3M_2022_10K,domain-relevant,Logical reasoning (based on numerical reasonin...,dg17,What drove operating margin change as of FY202...,Operating Margin for 3M in FY2022 has decrease...,None,OPEN_SOURCE,"[{'evidence_text': 'SG&A, measured as a percen...",Industrials,10k,2022,https://investors.3m.com/financials/sec-filing...
4,financebench_id_01865,3M,3M_2022_10K,novel-generated,None,None,"If we exclude the impact of M&A, which segment...",The consumer segment shrunk by 0.9% organically.,None,OPEN_SOURCE,[{'evidence_text': 'Worldwide Sales Change By...,Industrials,10k,2022,https://investors.3m.com/financials/sec-filing...


Overview of the data. 

In [11]:
row = df.iloc[7]
for col in ["company", "doc_name", "doc_type", "question", "answer", "doc_link"]:
    print(col, "→", row[col])


company → 3M
doc_name → 3M_2023Q2_10Q
doc_type → 10q
question → Does 3M maintain a stable trend of dividend distribution?
answer → Yes, not only they distribute the dividends on a routine basis, 3M has also been increasing the per share dividend for consecutive 65 years
doc_link → https://investors.3m.com/financials/sec-filings/content/0000066740-23-000058/0000066740-23-000058.pdf


In [4]:
ev = row["evidence"][0]
ev.keys()                         # evidence_text, doc_name, evidence_page_num, evidence_text_full_page
print(ev["evidence_text"])        # the exact snippet that answers the question
print(ev["evidence_text_full_page"])

This marked the 65th consecutive
year of dividend increases for 3M.
Table of Contents
Financial condition:
Refer to the section entitled Financial Condition and Liquidity later in MD&A for a discussion of items impacting cash flows.
In November 2018, 3Ms Board of Directors replaced the Companys February 2016 repurchase program with a new repurchase program. This new program authorizes the
repurchase of up to $10 billion of 3Ms outstanding common stock, with no pre-established end date. In the first six months of 2023, the Company purchased $29 million of its
own stock, compared to $773 million of stock purchases in the first six months of 2022. As of June 30, 2023, approximately $4.2 billion remained available under the
authorization. In February 2023, 3Ms Board of Directors declared a first-quarter 2023 dividend of $1.50 per share, an increase of 1 percent. This marked the 65th consecutive
year of dividend increases for 3M. In May 2023, 3M's Board of Directors declared a second-quarte

evidence section might be giving hints to how chunking and splitting has been done


In [ ]:
df["question_type"].value_counts()
df["doc_type"].value_counts()     # 10k, 10q, 8k, Earnings
df["doc_name"].nunique()          # 84 distinct filings
df["doc_name"].value_counts()     # questions per filing
df["evidence"].apply(len).value_counts()

lens = [len(e["evidence_text"]) for evs in df["evidence"] for e in evs]
pd.Series(lens).describe()   


evidence
1    115
2     31
3      4
Name: count, dtype: int64

checking basic statistics of the data, especially the evidence column

In [16]:
from pypdf import PdfReader
reader = PdfReader("sample pdf.pdf")
len(reader.pages)                    # how many pages
print(reader.pages[59].extract_text())  # compare with evidence_text_full_page

Table of Contents
Sales and operating income (loss) by business segment:
The following tables contain sales and operating income (loss) results by business segment for the three and six months ended June 30, 2023 and 2022. Refer to the section
entitled Performance by Business Segment later in MD&A for additional discussion concerning 2023 versus 2022 results, including Corporate and Unallocated. Refer to Note
15 for additional information on business segments.
Three months ended June 30,  
2023 2022 % change
(Dollars in millions)
Net
Sales
Operating
Income (Loss)
Net
Sales
Operating
Income (Loss)
Net
Sales
Operating
Income (Loss)
Business Segments
Safety and Industrial $ 2,765 $ 534 $ 2,924 $ (707) (5.5) % (175.6) %
Transportation and Electronics 2,191 410 2,268 475 (3.4) (13.8)
Health Care 2,075 411 2,179 492 (4.8) (16.4)
Consumer 1,293 235 1,330 248 (2.7) (5.1)
Corporate and Unallocated 1 (10,548) 1 (398)
Total Company $ 8,325 $ (8,958) $ 8,702 $ 110 (4.3) % N/M
Six months ended June